In [8]:
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(override=True)

DATA_PATH = os.getenv("DATA_PATH")

if DATA_PATH is None:
    raise ValueError("DATA_PATH is not defined in .env")

data_dir = Path(DATA_PATH)


# Load n2 dataset
df_n2 = pd.read_csv(
    data_dir / "df_n2.csv"
)


# Find all n3 generated files
files = list(data_dir.glob("*n3*.csv"))

print(f"Found {len(files)} files:")
for f in files:
    print(f" - {f.name}")


df_list = []

for file in files:

    try:
        df = pd.read_csv(file)

        # Skip empty dataframes
        if df.empty:
            print(f"[SKIPPED EMPTY] {file.name}")
            continue

        df["generation_file"] = file.name

        df_list.append(df)

    except pd.errors.EmptyDataError:
        print(f"[SKIPPED NO DATA] {file.name}")
        continue

    except Exception as e:
        print(f"[SKIPPED ERROR] {file.name}: {e}")
        continue


if df_list:

    df_combined = pd.concat(
        df_list,
        ignore_index=True
    )

    df_combined = df_combined.sort_values(
        by="patch_id"
    )

    print("Final shape:", df_combined.shape)

else:

    print("No valid CSV files found.")

    df_combined = pd.DataFrame()

Found 1 files:
 - df_n3_3.csv
Final shape: (13, 7)


In [9]:
# Rename generated comment column
df_combined = df_combined.rename(columns={'comment': 'generated_comment'})

# Select only the columns needed from df_n2 (human comments)
df_human = df_n2[['patch_id', 'comment']].rename(columns={'comment': 'human_comment'})

# Merge human comments + hunk + relevant_context using patch_id
df_combined = df_combined.merge(
    df_n2[
        [
            "patch_id",
            "comment",
            "hunk",
            "relevant_context",
            "relevant_same_file_code_hunks",
            "relevant_different_files_code_hunks",
        ]
    ].rename(
        columns={
            "comment": "human_comment",
        }
    ),
    on="patch_id",
    how="left",
)


In [10]:
df_combined.to_csv(f'{DATA_PATH}df_n3.csv', index=False)